# 🎧 AI DJ — Turn a Vibe into an Album Cover

**Assignment coverage:**
1. Set up a HuggingFace text-to-image pipeline and generate one test image to confirm it works
2. Invent 6 fictional music acts with different genres: lo-fi bedroom pop, dramatic classical, chaotic punk, mysterious ambient, cheesy 80s pop duo, and one made-up genre (rock'n'roll for kids)
3. Write a prompt for each act (style, colors, mood, imagery) and generate one image per act
4. Pick the best act and run its prompt 4 times — compare what stays the same and what changes
5. Try a deliberately terrible prompt (`"music"`) and document what the model does with nothing to work with
6. Curate the 3 favourite images and give each a fake album title and tracklist (interactive HTML gallery)

> ⚠️ **Runtime → Change runtime type → GPU (T4)** before running. The full run takes ~10–15 min on a T4.

In [ ]:
!pip install -q torch torchvision matplotlib numpy diffusers transformers accelerate

## Task 1 — HuggingFace pipeline + test image

We load **Stable Diffusion XL** in fp16 (`variant="fp16"` downloads the half-precision weights directly — half the bandwidth). Two helpers:
- `generate_cover()` — one 1024×1024 cover with a **fixed seed**, so every result in this notebook is reproducible,
- `show()` — quick matplotlib preview.

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import DiffusionPipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

pipeline = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to(device)

# Common negative prompt for the whole project
NEGATIVE_PROMPT = ("low quality, blurry, distorted, bad anatomy, "
                   "text, watermark, signature, deformed")

def generate_cover(filename, prompt, seed=42, steps=30, guidance=7.5,
                   negative_prompt=NEGATIVE_PROMPT):
    """Generate one 1024x1024 album cover with a fixed seed (reproducible)."""
    g = torch.Generator(device).manual_seed(seed)
    img = pipeline(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=guidance,
        width=1024, height=1024,
        generator=g,
    ).images[0]
    img.save(f"{filename}.png")
    return img

def show(img, title, size=6):
    plt.figure(figsize=(size, size))
    plt.imshow(img)
    plt.axis('off')
    plt.title(title, fontsize=10)
    plt.show()

In [ ]:
# Test image — confirms the pipeline works end to end
test_img = generate_cover("test_image", "Just basic dramatic classical cover")
show(test_img, 'Test: "Just basic dramatic classical cover"')
print("✅ Pipeline works — test image generated.")

## Tasks 2 + 3 — six fictional acts and their prompts

Each prompt explicitly encodes the four required elements:
**style** (e.g. *cinematic oil painting*), **colors** (a named palette), **mood** (e.g. *tempestuous, awe-inspiring*) and **imagery** (the central subject and environment).

In [ ]:
ACTS = {
    "lofi_bedroom_pop": {
        "artist": "Pillow Static",
        "album": "Sunday Reruns",
        "genre": "Lo-fi bedroom pop",
        "prompt": (
            "cozy teenage bedroom at dusk, cassette player and tangled fairy lights on a windowsill, "
            "rain on the glass, soft film grain, muted pastel palette of dusty pink, faded teal and warm amber, "
            "nostalgic dreamy mood, lo-fi anime-inspired illustration, gentle window light"
        ),
    },
    "dramatic_classical": {
        "artist": "The Velvet Requiem",
        "album": "Ashes of the Overture",
        "genre": "Dramatic classical",
        "prompt": (
            "grand piano engulfed by a violent ocean storm at night, shattered marble statues sinking in waves, "
            "dark romanticism, cinematic oil painting, dramatic chiaroscuro lighting, "
            "deep crimson, obsidian black and fractured gold palette, tempestuous awe-inspiring mood"
        ),
    },
    "chaotic_punk": {
        "artist": "Dumpster Ballet",
        "album": "Refund Denied",
        "genre": "Chaotic punk",
        "prompt": (
            "ripped xerox collage of a screaming crowd and a smashed guitar, safety pins and torn tape, "
            "harsh photocopy texture, DIY zine cut-and-paste style, "
            "acid yellow, black and blood red palette, aggressive anarchic mood, high contrast grain"
        ),
    },
    "mysterious_ambient": {
        "artist": "Halocline",
        "album": "Below the Thermocline",
        "genre": "Mysterious ambient",
        "prompt": (
            "vast empty fog-covered lake at 4am, a single faint light under the water surface, "
            "minimalist long-exposure photography, deep indigo, slate grey and pale cyan palette, "
            "eerie meditative mood, negative space, soft gradients, endless horizon"
        ),
    },
    "cheesy_80s_pop_duo": {
        "artist": "Neon Cousins",
        "album": "Call Me After Aerobics",
        "genre": "Cheesy 80s pop duo",
        "prompt": (
            "two confident pop singers in shoulder-padded metallic jackets, back to back pose, "
            "airbrushed 1980s studio portrait, laser grid background, "
            "hot pink, electric blue and chrome palette, glamorous over-the-top mood, soft glow, retro gloss"
        ),
    },
    "kids_rocknroll": {
        "artist": "The Wiggly Amps",
        "album": "Bedtime Is Cancelled!!",
        "genre": "Rock'n'roll for kids (made-up genre)",
        "prompt": (
            "cartoon animal band on a cardboard stage, dinosaur drummer and rabbit guitarist mid-jump, "
            "bold playful children's book illustration, thick outlines, confetti in the air, "
            "primary red, sunny yellow and sky blue palette, joyful energetic mood"
        ),
    },
}

for key, act in ACTS.items():
    print(f"🎵 {act['artist']} — \"{act['album']}\"  [{act['genre']}]")

In [ ]:
# One cover per act (6 generations, ~1 min each on T4)
cover_images = {}
for key, act in ACTS.items():
    print(f"Generating: {act['artist']} …")
    cover_images[key] = generate_cover(f"cover_{key}", act["prompt"])

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, (key, act) in zip(axes.flat, ACTS.items()):
    ax.imshow(cover_images[key])
    ax.axis('off')
    ax.set_title(f"{act['artist']}\n\"{act['album']}\"", fontsize=9)
plt.suptitle("6 fictional acts — one cover each", fontsize=13)
plt.tight_layout()
plt.show()

## Task 4 — best act, same prompt, 4 different seeds

We run the *dramatic classical* prompt with 4 fixed seeds. Compare the grid below and note what the **prompt** controls versus what the **seed** controls.

In [ ]:
BEST = "dramatic_classical"
seeds = [7, 42, 123, 999]
variations = []
for s in seeds:
    print(f"Seed {s} …")
    variations.append(generate_cover(f"best_{BEST}_seed{s}", ACTS[BEST]["prompt"], seed=s))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img, s in zip(axes, variations, seeds):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"seed={s}", fontsize=10)
plt.suptitle(f"{ACTS[BEST]['artist']} — same prompt, 4 different seeds", fontsize=13)
plt.tight_layout()
plt.show()

### Observations (task 4) — *edit this cell to match your actual results*

- **Stays the same** (controlled by the prompt): the grand piano as the central motif, the storm/ocean setting, the crimson–black–gold palette, the painterly chiaroscuro style, the overall dramatic mood.
- **Changes** (controlled by the seed): composition and camera angle, number and placement of the marble statues, the phase of the storm (crashing waves vs. mist), the gold-to-red balance, wave detail.

**Conclusion:** the prompt controls *content and style*; the seed controls the specific *composition*.

## Task 5 — a deliberately terrible prompt

One word, no style, no colors, no mood, no imagery. Note: we also drop the negative prompt here, so the model receives literally nothing but `"music"`.

In [ ]:
bad_img = generate_cover("bad_prompt", "music", negative_prompt="")
show(bad_img, 'Terrible prompt: "music"')

### Documentation (task 5) — *edit this cell to match your actual result*

With the prompt `"music"` the model has no clues about style, color, mood, or subject — so it falls back to the most frequent dataset associations for the word: typically notes, instruments, headphones, generic gradients, and often **the literal word "music" rendered as typography** (text is one of the strongest associations for single-word prompts). The composition tends to be generic or chaotic.

Compared side by side with any task-3 cover, this shows how much of the final quality comes from the structured prompt (style + colors + mood + imagery) rather than from the model alone.

## Task 6 — curated selection: 3 favourites + fake album titles + tracklists

The three picks are exported to a standalone **`gallery.html`** (vinyl-sleeve design, record slides out on hover) and previewed inline below. Images are embedded as base64, so the single HTML file works anywhere.

> Swap the entries in `FAVOURITES` if different covers turned out better in your run — just change the `file` names.

In [ ]:
import base64
from IPython.display import HTML, display

FAVOURITES = [
    {
        "file": f"best_{BEST}_seed42.png",
        "artist": "The Velvet Requiem",
        "album": "Ashes of the Overture",
        "genre": "Dramatic Classical",
        "tracks": ["Overture in Flames", "Marble Lungs", "Storm Sonata No. 9",
                   "Requiem for the Undertow", "Gilded Wreckage", "Final Curtain, First Light"],
    },
    {
        "file": "cover_mysterious_ambient.png",
        "artist": "Halocline",
        "album": "Below the Thermocline",
        "genre": "Mysterious Ambient",
        "tracks": ["Descent", "Four A.M. Water", "Pressure Hymn",
                   "The Light Below", "Sonar Ghosts", "Surface (Reprise)"],
    },
    {
        "file": "cover_cheesy_80s_pop_duo.png",
        "artist": "Neon Cousins",
        "album": "Call Me After Aerobics",
        "genre": "80s Pop",
        "tracks": ["Call Me After Aerobics", "Cassette Heart", "Leg Warmer Love",
                   "Payphone Serenade", "VHS Sunset", "Rewind Me"],
    },
]

def img_to_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

cards = ""
for i, fav in enumerate(FAVOURITES, 1):
    tracks_html = "".join(
        f"<li><span class='no'>{j:02d}</span>{t}</li>"
        for j, t in enumerate(fav["tracks"], 1)
    )
    cards += f"""
    <article class="sleeve">
      <div class="cover-wrap">
        <img src="data:image/png;base64,{img_to_b64(fav['file'])}" alt="{fav['album']}">
        <div class="vinyl"></div>
      </div>
      <div class="meta">
        <p class="genre">{fav['genre']} · LP {i:02d}</p>
        <h2>{fav['album']}</h2>
        <p class="artist">{fav['artist']}</p>
        <ol class="tracks">{tracks_html}</ol>
      </div>
    </article>"""

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>AI DJ — Curated Selection</title>
<link href="https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,600;1,500&family=Space+Grotesk:wght@400;500&display=swap" rel="stylesheet">
<style>
  :root {{
    --ink: #17141f; --paper: #efe8da; --gold: #c9a45c;
    --wine: #7a2c3b; --fog: #9a94a8;
  }}
  * {{ box-sizing: border-box; margin: 0; }}
  body {{
    background: var(--ink); color: var(--paper);
    font-family: 'Space Grotesk', sans-serif;
    padding: 56px 24px 80px;
  }}
  header {{ max-width: 1100px; margin: 0 auto 48px; }}
  header p.label {{
    letter-spacing: .35em; text-transform: uppercase;
    font-size: 11px; color: var(--gold); margin-bottom: 14px;
  }}
  header h1 {{
    font-family: 'Cormorant Garamond', serif;
    font-size: clamp(38px, 6vw, 64px);
    font-weight: 600; line-height: 1.05;
  }}
  header h1 em {{ font-style: italic; color: var(--gold); }}
  header p.sub {{ margin-top: 14px; color: var(--fog); max-width: 52ch; }}
  .rack {{
    max-width: 1100px; margin: 0 auto;
    display: grid; gap: 40px;
    grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
  }}
  .sleeve {{
    background: #1f1b29; border: 1px solid #2c2738;
    padding: 18px 18px 26px;
  }}
  .cover-wrap {{ position: relative; aspect-ratio: 1; }}
  .cover-wrap img {{
    position: relative; z-index: 2;
    width: 100%; aspect-ratio: 1; object-fit: cover; display: block;
    box-shadow: 0 12px 30px rgba(0,0,0,.55);
  }}
  /* signature: the vinyl record slides out from the sleeve on hover */
  .vinyl {{
    position: absolute; top: 4%; left: 0; z-index: 1;
    width: 92%; aspect-ratio: 1; border-radius: 50%;
    background: radial-gradient(circle at center,
      var(--gold) 0 12%, #0c0a10 13% 15%, #191521 16% 100%);
    box-shadow: inset 0 0 0 1px #000;
    transition: transform .5s cubic-bezier(.2,.7,.2,1);
  }}
  .sleeve:hover .vinyl {{ transform: translateX(26%) rotate(40deg); }}
  @media (prefers-reduced-motion: reduce) {{
    .vinyl {{ transition: none; }}
    .sleeve:hover .vinyl {{ transform: none; }}
  }}
  .meta {{ margin-top: 22px; }}
  .genre {{
    font-size: 10px; letter-spacing: .3em; text-transform: uppercase;
    color: var(--gold); margin-bottom: 8px;
  }}
  .meta h2 {{
    font-family: 'Cormorant Garamond', serif;
    font-size: 27px; font-weight: 600; line-height: 1.1;
  }}
  .artist {{ color: var(--fog); margin: 4px 0 16px; font-size: 14px; }}
  .tracks {{ list-style: none; padding: 0; border-top: 1px solid #2c2738; }}
  .tracks li {{
    display: flex; gap: 14px; align-items: baseline;
    padding: 7px 0; font-size: 13.5px;
    border-bottom: 1px solid #2c2738;
  }}
  .tracks .no {{ color: var(--wine); font-size: 11px; letter-spacing: .1em; }}
</style>
</head>
<body>
  <header>
    <p class="label">AI DJ · Curated selection</p>
    <h1>Three sleeves,<br><em>three worlds.</em></h1>
    <p class="sub">Three favourite covers generated by Stable Diffusion XL —
    each with a fictional album and tracklist. Hover over a cover to slide out the record.</p>
  </header>
  <section class="rack">{cards}</section>
</body>
</html>"""

with open("gallery.html", "w", encoding="utf-8") as f:
    f.write(html)

display(HTML(html))   # inline preview right here in Colab
print("💾 Saved gallery.html — downloadable from the Files panel on the left.")

## Optional — save all outputs to Google Drive

Run this cell if you want the covers and `gallery.html` copied to your Drive.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/AI_AlbumCovers'
os.makedirs(save_dir, exist_ok=True)

files = (["test_image.png", "bad_prompt.png", "gallery.html"]
         + [f"cover_{k}.png" for k in ACTS]
         + [f"best_{BEST}_seed{s}.png" for s in seeds])

for f in files:
    src = f"/content/{f}"
    if os.path.exists(src):
        shutil.copy(src, save_dir)
        print("Copied:", f)

print(f"All generated files copied to: {save_dir}")